### Full analysis

In [9]:
import os
import json
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, kendalltau, pearsonr

# --- Configuration & Path Resolution ---
SHOT_SETTINGS = [1, 5, 10, 15, 20, 25]

# Set evaluation source: 'lambda_tuned' or 'epoch_tuned'
# EVAL_SOURCE = "lambda_tuned"
EVAL_SOURCE = "epoch_tuned"

# Metric mapping: Display Name -> (Subfolder, JSON filename, JSON score key)
METRICS_MAP = {
    "LogME": ("logME", "all_models_logme.json", "overall_logme"),
    # "H-score": ("H_score", "all_models_hscore.json", "overall_hscore"),
    "LFC": ("LFC_score", "all_models_lfc.json", "overall_lfc"),
    "LGC": ("LGC_linear", "all_models_lgc.json", "overall_lgc"),
    # "RKHS": ("RKHS_linear", "all_models_rkhs.json", "overall_rkhs"),
    "LGC-CLS": ("LGC_cls_linear", "all_models_lgc_cls.json", "overall_lgc_cls"),
    "SVM-Margin": ("SVM_margin_linear", "all_models_svm_margin.json", "overall_svm_margin"),
    "SVM-LOO-Margin": ("SVM_margin_linear", "all_models_svm_margin.json", "overall_loo_margin"),
    # "LGC-CLS_new": ("LGC_cls_linear_new", "all_models_lgc_cls_new.json", "overall_lgc_cls_new"),
    # "SVM-Margin_new": ("SVM_margin_linear_new", "all_models_svm_margin_new.json", "overall_svm_margin_new"),
    # "SVM-LOO-Margin_new": ("SVM_loo_margin_linear_new", "all_models_svm_loo_margin_new.json", "overall_loo_margin_new")
}

# Automatically resolve workspace root whether run from root or a subfolder
base_prefix = "." if os.path.exists("model_scores") else ".."

eval_results_dir = os.path.join(
    base_prefix,
    "classification_models_test_results",
    f"linear_model_{EVAL_SOURCE}",
)
scores_root_dir = os.path.join(base_prefix, "model_scores")
output_summary_dir = os.path.join(base_prefix, "rank_correlation_results")
os.makedirs(output_summary_dir, exist_ok=True)


def locate_metric_file(scores_root: str, shots: int, metric_folder: str, json_filename: str) -> str | None:
    """Locates the JSON score file for a specific shot regime and metric."""
    path = os.path.join(
        scores_root,
        f"model_scores_{shots}_shot",
        metric_folder,
        json_filename,
    )
    return path if os.path.exists(path) else None


# --- Main Analysis Loop ---
consolidated_correlations = []
per_shot_detailed_data = {}

for shots in SHOT_SETTINGS:
    eval_file = os.path.join(eval_results_dir, f"linear_eval_{shots}_shot.json")

    if not os.path.exists(eval_file):
        print(f"Skipping {shots}-shot: Evaluation file not found at {eval_file}")
        continue

    # 1. Load downstream test accuracies
    with open(eval_file, "r") as f:
        eval_data = json.load(f)

    model_names = sorted(eval_data.keys())
    test_accuracies = [eval_data[m]["overall_accuracy"] for m in model_names]

    shot_df = pd.DataFrame({
        "Model": model_names,
        "Test_Accuracy": test_accuracies,
    })

    # 2. Load transferability metric scores
    metric_scores = {}
    for metric_name, (folder, json_file, score_key) in METRICS_MAP.items():
        metric_path = locate_metric_file(scores_root_dir, shots, folder, json_file)

        if metric_path is None:
            print(f"Warning: Score file for {metric_name} ({shots}-shot) not found.")
            metric_scores[metric_name] = [np.nan] * len(model_names)
            shot_df[metric_name] = metric_scores[metric_name]
            continue

        with open(metric_path, "r") as f:
            score_data = json.load(f)

        scores = [
            score_data[m].get(score_key, np.nan) if m in score_data else np.nan
            for m in model_names
        ]
        metric_scores[metric_name] = scores
        shot_df[metric_name] = scores

    per_shot_detailed_data[f"{shots}_shot"] = shot_df.to_dict(orient="records")

    # 3. Display per-shot scores and accuracies
    print(f"\n{'=' * 78}")
    print(f"{f'PERFORMANCE & METRIC VALUES ({shots}-SHOT | {EVAL_SOURCE})':^78}")
    print(f"{'=' * 78}")
    print(shot_df.to_string(index=False, justify="center"))

    # 4. Compute rank correlations
    print(f"\n{'-' * 78}")
    print(f"{f'RANK CORRELATION WITH TEST ACCURACY ({shots}-SHOT)':^78}")
    print(f"{'-' * 78}")
    print(f"{'Metric':<10} | {'Spearman ρ':>10} | {'p-val':>8} | {'Kendall τ':>10} | {'p-val':>8} | {'Pearson r':>10} | {'p-val':>8}")
    print(f"{'-' * 78}")

    accs = np.array(test_accuracies, dtype=np.float64)

    for metric_name in METRICS_MAP:
        vals = np.array(metric_scores[metric_name], dtype=np.float64)
        valid_mask = ~np.isnan(vals) & ~np.isnan(accs)

        if valid_mask.sum() >= 3:
            s_rho, s_pval = spearmanr(vals[valid_mask], accs[valid_mask])
            k_tau, k_pval = kendalltau(vals[valid_mask], accs[valid_mask])
            p_r, p_pval = pearsonr(vals[valid_mask], accs[valid_mask])

            print(
                f"{metric_name:<10} | {s_rho:>10.4f} | {s_pval:>8.4f} | "
                f"{k_tau:>10.4f} | {k_pval:>8.4f} | {p_r:>10.4f} | {p_pval:>8.4f}"
            )

            consolidated_correlations.append({
                "shots": shots,
                "metric": metric_name,
                "spearman_rho": float(s_rho),
                "spearman_pvalue": float(s_pval),
                "kendall_tau": float(k_tau),
                "kendall_pvalue": float(k_pval),
                "pearson_r": float(p_r),
                "pearson_pvalue": float(p_pval),
            })
        else:
            print(f"{metric_name:<10} | Insufficient valid data points for correlation.")


# --- Consolidated Multi-Shot Summary & Export ---
if consolidated_correlations:
    corr_df = pd.DataFrame(consolidated_correlations)

    pivot_spearman = corr_df.pivot(index="metric", columns="shots", values="spearman_rho")
    pivot_kendall = corr_df.pivot(index="metric", columns="shots", values="kendall_tau")
    pivot_pearson = corr_df.pivot(index="metric", columns="shots", values="pearson_r")

    print(f"\n{'=' * 78}")
    print(f"{f'CONSOLIDATED SPEARMAN RANK CORRELATION (ρ) [{EVAL_SOURCE}]':^78}")
    print(f"{'=' * 78}")
    print(pivot_spearman.round(4).to_string())

    print(f"\n{'=' * 78}")
    print(f"{f'CONSOLIDATED KENDALL TAU RANK CORRELATION (τ) [{EVAL_SOURCE}]':^78}")
    print(f"{'=' * 78}")
    print(pivot_kendall.round(4).to_string())

    print(f"\n{'=' * 78}")
    print(f"{f'CONSOLIDATED PEARSON CORRELATION (r) [{EVAL_SOURCE}]':^78}")
    print(f"{'=' * 78}")
    print(pivot_pearson.round(4).to_string())

    csv_file = os.path.join(output_summary_dir, f"rank_correlations_{EVAL_SOURCE}.csv")
    corr_df.to_csv(csv_file, index=False)

    json_file = os.path.join(output_summary_dir, f"rank_correlations_{EVAL_SOURCE}.json")
    with open(json_file, "w") as f:
        json.dump(
            {
                "eval_source": EVAL_SOURCE,
                "correlations": consolidated_correlations,
                "model_level_data": per_shot_detailed_data,
            },
            f,
            indent=4,
        )

    print(f"\nSaved correlation reports to:")
    print(f" - CSV:  {csv_file}")
    print(f" - JSON: {json_file}")


              PERFORMANCE & METRIC VALUES (1-SHOT | epoch_tuned)              
      Model         Test_Accuracy   LogME     LFC      LGC    LGC-CLS  SVM-Margin SVM-LOO-Margin
          AVES-bio    0.353333    0.929706 0.901207 0.901207 0.922730  -0.897185       None     
BirdAVES-biox-base    0.386667    1.121703 0.959781 0.959781 0.952631  -0.859808       None     
      DistilHuBERT    0.353333    1.215707 0.799697 0.799697 0.783486  -0.949937       None     
            HuBERT    0.326667    0.921762 0.833500 0.833500 0.804529  -0.926630       None     
       MusicHuBERT    0.233333    1.091213 0.924622 0.924622 0.812840  -0.876712       None     
             WavLM    0.273333    1.367226 0.778391 0.778391 0.750838  -0.978405       None     

------------------------------------------------------------------------------
                 RANK CORRELATION WITH TEST ACCURACY (1-SHOT)                 
------------------------------------------------------------------------------
Met